Hypothesis H-A: Subjective sleep quality (SSQ) and objective sleep quality (OSQ), represented individually by total sleep time (TST), sleep onset latency (SOL), wake after sleep onset (WASO), and sleep efficiency (SE), show meaningful correlation in a within-subject setting.

The hypothesis will be tested in a longitudinal, single-subject dataset.

**Please note that the analyses shown in this notebook were performed on synthetic data, and the results do not reflect the actual outcomes of the study.**

For more details, see the related preregistration at osf.io.

In [37]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import spearmanr, pearsonr
import matplotlib.dates as mdates

In [38]:
strength = 50  # a slider [0, 100] that determines the amount of randomness in data;
               # 0: more random, less correlation; 100: less random: more correlation
rng = np.random.default_rng(9)  # to keep things reproducible

In [39]:
start_date = "2000-01-01"
end_date   = "2003-12-30"
dates = pd.date_range(start=start_date, end=end_date, freq="D").strftime('%Y-%m-%d')
n = len(dates)

# Subjective Likert-scale sleep quality (1..5)
dataset_n1 = pd.DataFrame({
    "date": dates,
    "Sub_id": 1,
    "Subj_likert": rng.integers(1, 6, size=n)})

# Map Likert levels to slider bounds and sample a continuous subjective slider score [0,100]
bounds = {1: (0.0, 19.0), 2: (20.0, 39.0), 3: (40.0, 59.0), 4: (60.0, 79.99), 5: (80.0, 100.0)}
low  = np.vectorize(lambda x: bounds[int(x)][0])(dataset_n1["Subj_likert"].values)
high = np.vectorize(lambda x: bounds[int(x)][1])(dataset_n1["Subj_likert"].values)
dataset_n1["Subj_slide"] = np.round(rng.uniform(low, high), 2)

print("Mock QSci data:\n")
print(dataset_n1.iloc[:20, :].to_string(index=False))

Mock QSci data:

      date  Sub_id  Subj_likert  Subj_slide
2000-01-01       1            3       56.58
2000-01-02       1            5       91.30
2000-01-03       1            5       92.44
2000-01-04       1            2       34.23
2000-01-05       1            1        5.76
2000-01-06       1            4       69.49
2000-01-07       1            4       75.37
2000-01-08       1            4       60.92
2000-01-09       1            4       60.62
2000-01-10       1            4       65.96
2000-01-11       1            5       87.68
2000-01-12       1            5       82.54
2000-01-13       1            5       81.15
2000-01-14       1            5       85.58
2000-01-15       1            4       73.09
2000-01-16       1            5       83.70
2000-01-17       1            1        7.70
2000-01-18       1            1       12.64
2000-01-19       1            4       70.63
2000-01-20       1            3       45.75


In [40]:
# Helper to convert Likert to a quality score in [0,1]
def quality_from_likert(likert):
    return (likert - 1) / 4.0

# Generate objective sleep metrics with tunable correlation to subjective quality
def synthesize_metric(q, low_val, high_val, strength_0_100=0, positive=True, jitter=0.0):
    s = np.clip(strength_0_100 / 100.0, 0.0, 1.0)
    base_noise = rng.random(len(q))
    trend = q if positive else (1.0 - q)
    z = (1.0 - s) * base_noise + s * trend
    if jitter > 0:
        z = np.clip(z + rng.normal(0, jitter, size=len(z)), 0, 1)
    out = low_val + z * (high_val - low_val)
    return out

In [41]:
# Generating random values for objevtive metrics
target_q = quality_from_likert(dataset_n1["Subj_likert"].values)
# TST: minutes, higher is better. [~300–600]
dataset_n1["TST"] = synthesize_metric(target_q, low_val=300, high_val=600,
                                      strength_0_100=strength, positive=True, jitter=0.01).round().astype(int)

# SE: proportion, higher is better. [~0.60–0.98].
dataset_n1["SE"] = np.round(
    synthesize_metric(target_q, low_val=0.50, high_val=1,
                      strength_0_100=strength, positive=True, jitter=0.01),3)

# SOL: minutes, lower is better. [~30–100].
dataset_n1["SOL"] = synthesize_metric(target_q, low_val=30, high_val=100,
                                      strength_0_100=strength, positive=False, jitter=0.01).round().astype(int)

# WASO: minutes, lower is better.[~10–100].
dataset_n1["WASO"] = synthesize_metric(target_q, low_val=10, high_val=100,
                                       strength_0_100=strength, positive=False, jitter=0.01).round().astype(int)
print("Mock QSci data:\n")
print(dataset_n1.iloc[0:10, :].to_string(index=False))

Mock QSci data:

      date  Sub_id  Subj_likert  Subj_slide  TST    SE  SOL  WASO
2000-01-01       1            3       56.58  501 0.855   50    46
2000-01-02       1            5       91.30  582 0.888   43    45
2000-01-03       1            5       92.44  559 0.972   64    10
2000-01-04       1            2       34.23  442 0.671   78    87
2000-01-05       1            1        5.76  365 0.601   67    59
2000-01-06       1            4       69.49  437 0.793   66    35
2000-01-07       1            4       75.37  533 0.890   48    63
2000-01-08       1            4       60.92  464 0.789   41    56
2000-01-09       1            4       60.62  510 0.763   71    38
2000-01-10       1            4       65.96  422 0.790   44    29


In [42]:
# Fisher z-transform CI for Pearson's r (assumes independence; OK for weekly; daily CI additionally via block bootstrap)
def fisher_ci(r, n, alpha=0.05):
    if n < 4 or np.isclose(1-abs(r), 0):
        return (np.nan, np.nan)
    z = np.arctanh(np.clip(r, -0.999999, 0.999999))
    se = 1 / np.sqrt(n - 3)
    z_lo = z - 1.96 * se
    z_hi = z + 1.96 * se
    return (np.tanh(z_lo), np.tanh(z_hi))

# Simple moving block bootstrap CI for correlation (serial-aware for daily)
rng_boot = np.random.default_rng(42)
def block_boot_ci(x, y, method="spearman", block=7, reps=2000, alpha=0.05):
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    ok = ~np.isnan(x) & ~np.isnan(y)
    x, y = x[ok], y[ok]
    n = len(x)
    if n == 0:
        return (np.nan, np.nan, np.nan)
    if n < max(3, 2*block):
        block = 1

    def draw_idx():
        out = []
        k = 0
        while k < n:
            start = rng_boot.integers(0, n - block + 1)
            out.append(np.arange(start, start + block))
            k += block
        return np.concatenate(out)[:n]

    if method == "spearman":
        corr = lambda a, b: spearmanr(a, b).correlation
    else:
        corr = lambda a, b: pearsonr(a, b)[0]

    stat_hat = corr(x, y)
    boot = np.empty(reps)
    for b in range(reps):
        idx = draw_idx()
        boot[b] = corr(x[idx], y[idx])

    lo, hi = np.percentile(boot, [100*alpha/2, 100*(1-alpha/2)])
    return float(stat_hat), float(lo), float(hi)

In [62]:
def outcome(df, level_label, threshold=0.20):
    def label_target(t):
        t_str = str(t).lower()
        if "slide" in t_str:  return "SSQ (slider)"
        if "likert" in t_str: return "SSQ (Likert)"
        return str(t)
    methods = [
        ("Spearman", "spearman_rho", "spearman_ci_lo", "spearman_ci_hi", "ρ"),
        ("Pearson",  "pearson_r",    "pearson_ci_lo",   "pearson_ci_hi",  "r")]

    have = {m[0]: set(m[1:4]).issubset(df.columns) for m in methods}

    for _, row in df.iterrows():
        metric = row.get("metric", "metric")
        target = label_target(row.get("target", "target"))
        for name, eff_col, lo_col, hi_col, sym in methods:
            if not have[name]:
                continue
            eff = float(row[eff_col]); lo = float(row[lo_col]); hi = float(row[hi_col])
            ci_excludes_zero = (lo > 0) or (hi < 0)
            meets_thresh = abs(eff) >= threshold
            accepted = ci_excludes_zero and meets_thresh
            print(
                f"{name} correlation between ({level_label}) {metric} and {target} "
                f"was {eff:.3f} (95% CI: {lo:.3f}–{hi:.3f}); "
                f"|{sym}|≥{threshold:.2f} is {'met' if meets_thresh else 'not met'}; "
                f"CI excludes 0 is {'yes' if ci_excludes_zero else 'no'}; "
                f"{'Hypothesis is accepted' if accepted else 'Hypothesis is rejected'}.")

In [44]:
# del dataset_n1

In [45]:
df = dataset_n1.copy()
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)
metrics = ["TST", "SE", "SOL", "WASO"]
targets = ["Subj_slide", "Subj_likert"]

In [46]:
# Make column names tidy and discover what actually exists
df.columns = df.columns.str.strip()

# Define expected names, then keep only those that exist in df
metrics_all = ["TST", "SE", "SOL", "WASO"]
targets_all = ["Subj_slide", "Subj_scale", "Subj_likert"]  # include both slider spellings

metrics  = [c for c in metrics_all if c in df.columns]
targets  = [c for c in targets_all if c in df.columns]

if not metrics or not targets:
    raise ValueError(f"No usable metrics/targets found. df has: {list(df.columns)}")

In [47]:
# Daily correlations (one row per metric × target):

daily_rows = []
for metric in metrics:
    for target in targets:
        # Spearman
        rho_hat, rho_lo, rho_hi = block_boot_ci(df[metric], df[target], method="spearman", block=7, reps=2000)
        # Pearson
        r_hat, r_lo, r_hi = block_boot_ci(df[metric], df[target], method="pearson", block=7, reps=2000)
        daily_rows.append({
            "metric": metric,
            "target": target,
            "n_obs": int(df[[metric, target]].dropna().shape[0]),
            "spearman_rho": rho_hat,
            "spearman_ci_lo": rho_lo,
            "spearman_ci_hi": rho_hi,
            "pearson_r": r_hat,
            "pearson_ci_lo": r_lo,
            "pearson_ci_hi": r_hi
        })

daily_results = pd.DataFrame(daily_rows)
daily_results[["spearman_rho","spearman_ci_lo","spearman_ci_hi",
               "pearson_r","pearson_ci_lo","pearson_ci_hi"]] = daily_results[
    ["spearman_rho","spearman_ci_lo","spearman_ci_hi","pearson_r","pearson_ci_lo","pearson_ci_hi"]].round(4)

print("\nDaily correlations with 95% CI (block bootstrap):\n")
print(daily_results.to_string(index=False))


Daily correlations with 95% CI (block bootstrap):

metric      target  n_obs  spearman_rho  spearman_ci_lo  spearman_ci_hi  pearson_r  pearson_ci_lo  pearson_ci_hi
   TST  Subj_slide   1460        0.7615          0.7419          0.7780     0.7605         0.7427         0.7777
   TST Subj_likert   1460        0.7747          0.7570          0.7907     0.7713         0.7543         0.7877
    SE  Subj_slide   1460        0.7459          0.7252          0.7642     0.7461         0.7263         0.7625
    SE Subj_likert   1460        0.7568          0.7363          0.7742     0.7563         0.7394         0.7724
   SOL  Subj_slide   1460       -0.7527         -0.7728         -0.7315    -0.7539        -0.7729        -0.7354
   SOL Subj_likert   1460       -0.7756         -0.7943         -0.7561    -0.7732        -0.7910        -0.7556
  WASO  Subj_slide   1460       -0.7555         -0.7740         -0.7362    -0.7561        -0.7737        -0.7378
  WASO Subj_likert   1460       -0.7712     

In [63]:
outcome(daily_results, "Daily", threshold=0.20)

Spearman correlation between (Daily) TST and SSQ (slider) was 0.761 (95% CI: 0.742–0.778); |ρ|≥0.20 is met; CI excludes 0 is yes; Hypothesis is accepted.
Pearson correlation between (Daily) TST and SSQ (slider) was 0.760 (95% CI: 0.743–0.778); |r|≥0.20 is met; CI excludes 0 is yes; Hypothesis is accepted.
Spearman correlation between (Daily) TST and SSQ (Likert) was 0.775 (95% CI: 0.757–0.791); |ρ|≥0.20 is met; CI excludes 0 is yes; Hypothesis is accepted.
Pearson correlation between (Daily) TST and SSQ (Likert) was 0.771 (95% CI: 0.754–0.788); |r|≥0.20 is met; CI excludes 0 is yes; Hypothesis is accepted.
Spearman correlation between (Daily) SE and SSQ (slider) was 0.746 (95% CI: 0.725–0.764); |ρ|≥0.20 is met; CI excludes 0 is yes; Hypothesis is accepted.
Pearson correlation between (Daily) SE and SSQ (slider) was 0.746 (95% CI: 0.726–0.762); |r|≥0.20 is met; CI excludes 0 is yes; Hypothesis is accepted.
Spearman correlation between (Daily) SE and SSQ (Likert) was 0.757 (95% CI: 0.736

In [49]:
# Weekly aggregation
weekly = (
    df.set_index("date")
      .resample("W-MON")  # weeks ending on Monday (same as original)
      .agg({
          "TST": "mean",
          "SE": "mean",
          "SOL": "mean",
          "WASO": "mean",
          "Subj_slide": "mean",
          "Subj_likert": "median"  # median for Likert scale
      }))
weekly = weekly.dropna(how="all").reset_index()
print(weekly.iloc[0:10, :].to_string(index=False))

      date        TST       SE       SOL      WASO  Subj_slide  Subj_likert
2000-01-03 547.333333 0.905000 52.333333 33.666667   80.106667          5.0
2000-01-10 453.285714 0.756714 59.285714 52.428571   53.192857          4.0
2000-01-17 476.428571 0.822286 52.285714 29.714286   71.634286          5.0
2000-01-24 457.142857 0.797429 65.285714 64.714286   51.512857          3.0
2000-01-31 421.857143 0.725571 74.000000 54.285714   45.501429          2.0
2000-02-07 476.428571 0.771857 59.857143 40.571429   69.095714          4.0
2000-02-14 492.285714 0.719571 57.571429 42.714286   63.871429          4.0
2000-02-21 501.714286 0.848714 54.571429 41.714286   71.924286          5.0
2000-02-28 483.714286 0.792000 61.285714 55.714286   63.754286          4.0
2000-03-06 414.000000 0.729714 72.428571 45.571429   46.480000          3.0


In [50]:
# Weekly correlations

weekly_rows = []
for metric in metrics:
    for target in targets:
        # Spearman
        rho_hat_w, rho_lo_w, rho_hi_w = block_boot_ci(weekly[metric], weekly[target],
                                                      method="spearman", block=1, reps=1000)
        # Pearson
        mask = weekly[[metric, target]].dropna()
        r_w = pearsonr(mask[metric], mask[target])[0] if len(mask) >= 3 else np.nan
        lo_w, hi_w = fisher_ci(r_w, len(mask)) if len(mask) >= 4 else (np.nan, np.nan)
        weekly_rows.append({
            "metric": metric,
            "target": target,
            "n_weeks": int(mask.shape[0]),
            "spearman_rho": rho_hat_w,
            "spearman_ci_lo": rho_lo_w,
            "spearman_ci_hi": rho_hi_w,
            "pearson_r": r_w,
            "pearson_ci_lo": lo_w,
            "pearson_ci_hi": hi_w
        })

weekly_results = pd.DataFrame(weekly_rows)
weekly_results[["spearman_rho","spearman_ci_lo","spearman_ci_hi",
                "pearson_r","pearson_ci_lo","pearson_ci_hi"]] = weekly_results[
    ["spearman_rho","spearman_ci_lo","spearman_ci_hi","pearson_r","pearson_ci_lo","pearson_ci_hi"]].round(4)

print("\nWeekly correlations with 95% CI (Spearman via bootstrap; Pearson via Fisher z):\n")
print(weekly_results.to_string(index=False))



Weekly correlations with 95% CI (Spearman via bootstrap; Pearson via Fisher z):

metric      target  n_weeks  spearman_rho  spearman_ci_lo  spearman_ci_hi  pearson_r  pearson_ci_lo  pearson_ci_hi
   TST  Subj_slide      210        0.7053          0.6217          0.7728     0.7476         0.6812         0.8019
   TST Subj_likert      210        0.6650          0.5873          0.7418     0.6844         0.6051         0.7503
    SE  Subj_slide      210        0.7506          0.6758          0.8098     0.7645         0.7017         0.8154
    SE Subj_likert      210        0.6212          0.5167          0.7020     0.6357         0.5474         0.7100
   SOL  Subj_slide      210       -0.7250         -0.7868         -0.6487    -0.7405        -0.7961        -0.6726
   SOL Subj_likert      210       -0.6191         -0.6979         -0.5241    -0.6426        -0.7157        -0.5555
  WASO  Subj_slide      210       -0.7300         -0.7994         -0.6394    -0.7659        -0.8166        -0.703

In [64]:
outcome(weekly_results, "Weekly", threshold=0.20)

Spearman correlation between (Weekly) TST and SSQ (slider) was 0.705 (95% CI: 0.622–0.773); |ρ|≥0.20 is met; CI excludes 0 is yes; Hypothesis is accepted.
Pearson correlation between (Weekly) TST and SSQ (slider) was 0.748 (95% CI: 0.681–0.802); |r|≥0.20 is met; CI excludes 0 is yes; Hypothesis is accepted.
Spearman correlation between (Weekly) TST and SSQ (Likert) was 0.665 (95% CI: 0.587–0.742); |ρ|≥0.20 is met; CI excludes 0 is yes; Hypothesis is accepted.
Pearson correlation between (Weekly) TST and SSQ (Likert) was 0.684 (95% CI: 0.605–0.750); |r|≥0.20 is met; CI excludes 0 is yes; Hypothesis is accepted.
Spearman correlation between (Weekly) SE and SSQ (slider) was 0.751 (95% CI: 0.676–0.810); |ρ|≥0.20 is met; CI excludes 0 is yes; Hypothesis is accepted.
Pearson correlation between (Weekly) SE and SSQ (slider) was 0.764 (95% CI: 0.702–0.815); |r|≥0.20 is met; CI excludes 0 is yes; Hypothesis is accepted.
Spearman correlation between (Weekly) SE and SSQ (Likert) was 0.621 (95% CI

**#######################################################**

In [52]:
# Completely random data, no seed

start_date = "2000-01-01"
end_date = "2003-12-31"
rng = np.random.default_rng(9)

# Create date range and random data

n = len(dates)

qsci = pd.DataFrame({
    "date": dates,
    "Sub_id": 1,
    "TST": rng.integers(300, 601, size=n),
    "SE": np.round(rng.random(n), 3),
    "SOL": rng.integers(10, 101, size=n),
    "WASO": rng.integers(10, 101, size=n),
    "Subj": rng.integers(1, 6, size=n)
})
df = qsci